# Shadow手FK可视化

这个notebook用于可视化Shadow手的正向运动学(FK)结果，包括mesh、关键点和拟合球体。

## 功能
- 加载Shadow手URDF
- 计算正向运动学得到全局姿态
- 可视化所有link的mesh
- 显示关键点
- 显示拟合球体
- 支持关节角度调整

In [1]:
# 导入必要的库
import sys
import os

import numpy as np
import pyvista as pv
import yourdfpy
import trimesh
import jax.numpy as jnp
import jaxlie
from typing import Dict, List, Tuple
from interactive_gripper_tool.sphere_fitting import load_link_spheres
import json
import ipywidgets as widgets
from IPython.display import display
import pyroki as pk

In [2]:
# 设置文件路径
URDF_PATH = "test_assets/shadow/shadow_hand_right.urdf"
KEYPOINTS_PATH = "test_assets/shadow/shadow_hand_right_keypoints.json"
SPHERES_PATH = "test_assets/shadow/shadow_hand_right_spheres.json"

In [3]:
def load_hand_with_fk(urdf_path: str, joint_values: Dict[str, float] = None) -> Tuple[Dict[str, pv.PolyData], Dict[str, np.ndarray]]:
    """
    加载Shadow手，计算FK，应用全局变换到meshes
    
    Args:
        urdf_path: URDF文件路径
        joint_values: 关节值字典 {joint_name: value}
    
    Returns:
        (meshes_dict, poses_dict): meshes和全局位姿
    """
    print(f"加载URDF: {urdf_path}")
    
    # 1. 加载URDF和创建pyroki机器人
    urdf_obj = yourdfpy.URDF.load(urdf_path)
    robot = pk.Robot.from_urdf(urdf_obj)
    
    # 2. 设置关节值（默认零位姿）
    if joint_values is None:
        joint_values = {}
    actuated_names = urdf_obj.actuated_joint_names
    for joint_name in actuated_names:
        if joint_name not in joint_values:
            joint_values[joint_name] = 0.0
    
    # 3. 计算FK
    cfg_array = jnp.array([joint_values[name] for name in actuated_names])
    link_poses_rel_root = robot.forward_kinematics(cfg_array)
    
    # 4. 加载meshes并应用全局变换
    trimesh_scene = urdf_obj.scene
    if trimesh_scene is None:
        raise ValueError("无法加载URDF场景")
    
    meshes_dict = {}
    poses_dict = {}
    
    scene_graph = trimesh_scene.graph
    transform_graph = scene_graph.transforms
    all_node_data = scene_graph.transforms.node_data
    geometry_dict = trimesh_scene.geometry
    
    for link_name in scene_graph.nodes:
        if link_name == 'world':
            continue
        if link_name not in transform_graph.nodes:
            continue
        
        # 获取link的全局位姿
        if link_name in robot.links.names:
            link_idx = robot.links.names.index(link_name)
            T_world_link = jaxlie.SE3(link_poses_rel_root[link_idx])
            global_pose = np.array(T_world_link.as_matrix())
        else:
            # 对于不在pyroki中的links，使用单位矩阵
            global_pose = np.eye(4)
        
        poses_dict[link_name] = global_pose
        
        # 加载link的meshes
        link_meshes = []
        child_nodes = [to_node for from_node, to_node in transform_graph.edge_data if from_node == link_name]
        
        for child_node_name in child_nodes:
            if child_node_name in all_node_data and "geometry" in all_node_data[child_node_name]:
                geom_key = all_node_data[child_node_name]["geometry"]
                if geom_key not in geometry_dict:
                    continue
                
                trimesh_geom = geometry_dict[geom_key]
                local_transform = scene_graph.get(child_node_name, link_name)[0]
                
                if hasattr(trimesh_geom, 'to_mesh'):
                    trimesh_mesh = trimesh_geom.to_mesh()
                else:
                    trimesh_mesh = trimesh_geom.copy()
                
                # 应用局部变换 + 全局FK变换
                full_transform = global_pose @ local_transform
                trimesh_mesh.apply_transform(full_transform)
                link_meshes.append(trimesh_mesh)
        
        if not link_meshes:
            continue
        
        # 合并meshes
        if len(link_meshes) > 1:
            combined_mesh = trimesh.util.concatenate(link_meshes)
        else:
            combined_mesh = link_meshes[0]
        
        # 转换为PyVista
        pv_mesh = pv.wrap(combined_mesh)
        meshes_dict[link_name] = pv_mesh
    
    print(f"加载了 {len(meshes_dict)} 个link meshes，计算了 {len(poses_dict)} 个全局位姿")
    return meshes_dict, poses_dict

In [4]:
def load_keypoints(keypoints_path: str) -> Dict[str, np.ndarray]:
    """
    加载关键点数据
    
    Args:
        keypoints_path: 关键点文件路径
    
    Returns:
        keypoints_dict: {link_name: keypoints_array}
    """
    with open(keypoints_path, 'r') as f:
        data = json.load(f)
    
    keypoints_dict = {}
    for link_name, keypoints_list in data.items():
        keypoints_dict[link_name] = np.array(keypoints_list)
    
    print(f"加载了 {len(keypoints_dict)} 个link的关键点")
    return keypoints_dict

In [5]:
def visualize_hand(meshes_dict: Dict[str, pv.PolyData], 
                  keypoints_dict: Dict[str, np.ndarray],
                  spheres_dict: Dict[str, List[Tuple[np.ndarray, float]]],
                  poses_dict: Dict[str, np.ndarray] = None):
    """
    可视化整个手、关键点和球体
    
    Args:
        meshes_dict: {link_name: mesh}
        keypoints_dict: {link_name: keypoints_array}
        spheres_dict: {link_name: [(center, radius), ...]}
        poses_dict: {link_name: 4x4 pose matrix} (可选)
    """
    # 创建plotter
    plotter = pv.Plotter(notebook=True)
    plotter.set_background('white')
    
    # 设置相机
    plotter.camera_position = [(0.5, 0.5, 0.5), (0, 0, 0), (0, 0, 1)]
    
    # 颜色映射
    colors = ['lightblue', 'lightgreen', 'lightcoral', 'lightyellow', 'lightpink',
              'lightcyan', 'lavender', 'honeydew', 'aliceblue', 'beige']
    
    # 添加meshes
    for i, (link_name, mesh) in enumerate(meshes_dict.items()):
        color = colors[i % len(colors)]
        plotter.add_mesh(mesh, color=color, opacity=0.8, label=f'{link_name}')
    
    # 添加关键点
    for link_name, keypoints in keypoints_dict.items():
        if link_name in meshes_dict:
            # 将关键点变换到全局坐标（如果有poses_dict）
            if poses_dict and link_name in poses_dict:
                pose = poses_dict[link_name]
                # 关键点已经是全局坐标，这里不需要额外变换
                pass
            
            plotter.add_points(keypoints, color='red', point_size=8, label=f'{link_name} keypoints')
    
    # 添加球体
    sphere_count = 0
    for link_name, spheres in spheres_dict.items():
        for center, radius in spheres:
            # 创建球体mesh
            sphere_mesh = pv.Sphere(radius=radius, center=center)
            plotter.add_mesh(sphere_mesh, color='orange', opacity=0.4, label=f'{link_name} spheres' if sphere_count == 0 else None)
            sphere_count += 1
    
    # 添加坐标轴
    plotter.add_axes()
    
    # 添加图例
    plotter.add_legend()
    
    # 显示
    plotter.show()

In [6]:
# 加载数据
print("=== 加载Shadow手数据 ===")

# 1. 加载手和计算FK
meshes_dict, poses_dict = load_hand_with_fk(URDF_PATH)

# 2. 加载关键点
keypoints_dict = load_keypoints(KEYPOINTS_PATH)

# 3. 加载球体
spheres_dict = load_link_spheres(SPHERES_PATH)

print(f"\n数据统计:")
print(f"  Links: {len(meshes_dict)}")
print(f"  总关键点数: {sum(len(kp) for kp in keypoints_dict.values())}")
print(f"  总球体数: {sum(len(spheres) for spheres in spheres_dict.values())}")

=== 加载Shadow手数据 ===
加载URDF: test_assets/shadow/shadow_hand_right.urdf


2025-10-31 15:26:21.929 | INFO     | pyroki._robot_urdf_parser:_topologically_sort_joints:198 - Joints were not in topological order; they will be internally sorted.


加载了 24 个link meshes，计算了 57 个全局位姿
加载了 23 个link的关键点

数据统计:
  Links: 24
  总关键点数: 190
  总球体数: 31


In [7]:
# 可视化
print("=== 可视化 ===")
visualize_hand(meshes_dict, keypoints_dict, spheres_dict, poses_dict)

=== 可视化 ===


Widget(value='<iframe src="http://localhost:37697/index.html?ui=P_0x7a55673a21a0_0&reconnect=auto" class="pyvi…

## 关节控制

下面的代码允许你调整关节角度并重新可视化手的状态。

In [8]:
# 加载URDF获取关节信息
urdf_obj = yourdfpy.URDF.load(URDF_PATH)
joint_names = urdf_obj.actuated_joint_names

print(f"可控制关节: {len(joint_names)}")
for i, name in enumerate(joint_names):
    print(f"  {i}: {name}")

# 创建关节角度滑块
joint_sliders = {}
for name in joint_names:
    joint_sliders[name] = widgets.FloatSlider(
        value=0.0,
        min=-1.57,  # -90度
        max=1.57,   # +90度
        step=0.01,
        description=f'{name}:'
    )

# 显示控制界面
controls = widgets.VBox(list(joint_sliders.values()))
display(controls)

可控制关节: 24
  0: WRJ2
  1: WRJ1
  2: FFJ4
  3: FFJ3
  4: FFJ2
  5: FFJ1
  6: MFJ4
  7: MFJ3
  8: MFJ2
  9: MFJ1
  10: RFJ4
  11: RFJ3
  12: RFJ2
  13: RFJ1
  14: LFJ5
  15: LFJ4
  16: LFJ3
  17: LFJ2
  18: LFJ1
  19: THJ5
  20: THJ4
  21: THJ3
  22: THJ2
  23: THJ1


In [9]:
def update_visualization():
    """根据当前关节角度更新可视化"""
    # 获取当前关节角度
    joint_values = {name: slider.value for name, slider in joint_sliders.items()}
    
    # 重新计算FK
    meshes_dict_new, poses_dict_new = load_hand_with_fk(URDF_PATH, joint_values)
    
    # 可视化
    print(f"关节角度: {joint_values}")
    visualize_hand(meshes_dict_new, keypoints_dict, spheres_dict, poses_dict_new)

# 添加更新按钮
update_button = widgets.Button(description="更新可视化")
update_button.on_click(lambda b: update_visualization())
display(update_button)

Button(description='更新可视化', style=ButtonStyle())

## 使用说明

1. **基本可视化**: 运行上面的代码块可以看到手的默认姿态
2. **关节控制**: 调整滑块中的关节角度，然后点击"更新可视化"按钮
3. **数据说明**:
   - **蓝色系mesh**: 手的各个link
   - **红色点**: 关键点
   - **橙色半透明球**: 碰撞检测球体

## 注意事项

- 关键点和球体位置是基于默认姿态计算的
- 关节角度范围限制在±90度
- mesh显示为半透明以便观察内部结构